# Data Visualisation
## Task 2.2.1 Correctness, Plot Design and Clarity
## Task 2.2.2 Interesting Point Annotation

This notebook connects to MongoDB and **continuously polls** for new violation records,
updating four live charts every `POLL_INTERVAL` seconds as new streaming data arrives.

**Real-time approach:** The dashboard queries MongoDB on every poll cycle so that plots
always reflect the **latest data populated to the database** by the Spark Structured Streaming
application.

**Visualisations produced (live-updating):**
1. Daily Violation Count over Arrival Time — INSTANTANEOUS vs AVERAGE
2. Average Violation Speed over Arrival Time — with speed-limit reference lines
3. Violation Count by Camera and Type — grouped bar chart
4. Hourly Heatmap — hour-of-day × violation-type

**Data source:** MongoDB `traffic_monitoring.violations` collection (populated by the streaming application).

**How to run:**
1. Ensure MongoDB container is running and the streaming notebook has been executed.
2. Run all cells in order.
3. The final cell polls MongoDB every `POLL_INTERVAL` seconds and live-updates the dashboard.
4. Interrupt the kernel (`Stop`) to halt the loop.

## Step 1 Imports

In [ ]:
%matplotlib notebook

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from pymongo import MongoClient
from time import sleep

HOST_IP       = "host.docker.internal"
POLL_INTERVAL = 5   # seconds between MongoDB refreshes
COLORS        = {"INSTANTANEOUS": "#3b5bdb", "AVERAGE": "#e67700"}
# Reference lines only — actual violation detection uses camera metadata from camera.csv
REFERENCE_SPEED_LIMITS = {"INSTANTANEOUS": 110, "AVERAGE": 90}

print("Imports complete.")

## Step 2 MongoDB Data Loader

In [ ]:
def load_from_mongo():
    """Query MongoDB and return a flat DataFrame of all violation records."""
    client = MongoClient(host=HOST_IP, port=27017, serverSelectionTimeoutMS=3000)
    col = client["traffic_monitoring"]["violations"]
    rows = []
    for doc in col.find():
        for v in doc.get("violations", []):
            rows.append({
                "car_plate":        doc["car_plate"],
                "date":             doc["date"],
                "violation_type":   v["violation_type"],
                "camera_id_start":  v["camera_id_start"],
                "camera_id_end":    v["camera_id_end"],
                "timestamp_start":  v["timestamp_start"],
                "speed_reading":    v["speed_reading"],
            })
    client.close()
    return pd.DataFrame(rows)

print("load_from_mongo() defined.")

## Step 3 Initialise Live Dashboard

Four subplots are created once; each poll cycle clears and redraws them with the latest data.
All time-based plots (Plot 1 and Plot 2) use the **`timestamp_start` arrival timestamp**
recorded by the Spark Structured Streaming application, aggregated by day and hour.

**Speed limit reference lines** (Plot 2) are shown as visual guides only, based on the assignment specification:
- **INSTANTANEOUS** violations: Camera 1 → Camera 2 segment (speed limit **110 km/h**)
- **AVERAGE** violations: Camera 2 → Camera 3 segment (speed limit **90 km/h**)

*Note: These limits are used only as visual reference lines in this notebook. Actual violation detection is performed in the streaming application using camera metadata loaded from `camera.csv`.*

### Plot Descriptions and Operational Value

| Plot | What it shows | Operational question answered |
|------|--------------|-------------------------------|
| 1 Daily Count | Violations per day over arrival time, split by type | *Which days need extra patrol resources?* |
| 2 Speed Trend | 7-day rolling avg speed of violations over arrival time | *Is driver behaviour deteriorating over time?* |
| 3 Camera Breakdown | Violations per camera per type | *Which camera checkpoint is the highest-risk location?* |
| 4 Hourly Heatmap | Count by hour-of-day (arrival time) × violation type | *What time should mobile enforcement be deployed?* |

In [ ]:
def init_plots():
    """Create the figure and four subplots once."""
    fig = plt.figure(figsize=(14, 9))
    fig.suptitle("Real-Time Traffic Violation Dashboard", fontsize=13, fontweight="bold")
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.5, wspace=0.38)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    fig.show()
    fig.canvas.draw()
    return fig, ax1, ax2, ax3, ax4

print("init_plots() defined.")

## Step 4 Update Function

Called on every poll cycle. Loads fresh data from MongoDB, redraws all four plots,
and annotates notable points (MAX, MIN, SPIKE, PEAK HOUR).

In [ ]:
def update_plots(fig, ax1, ax2, ax3, ax4):
    """Reload MongoDB data and refresh all four subplots."""

    # Load & preprocess 
    df = load_from_mongo()
    if df.empty:
        print("No violation records in MongoDB yet — waiting...")
        return

    df["timestamp_start"] = pd.to_datetime(df["timestamp_start"])
    df["day"]             = df["timestamp_start"].dt.floor("D")
    df["hour_of_day"]     = df["timestamp_start"].dt.hour

    daily_counts = (
        df.groupby(["day", "violation_type"]).size()
        .reset_index(name="count")
    )
    daily_speed = (
        df.groupby(["day", "violation_type"])["speed_reading"].mean()
        .reset_index(name="avg_speed")
    )
    daily_pivot = (
        daily_counts.pivot(index="day", columns="violation_type", values="count")
        .fillna(0)
    )
    speed_pivot = (
        daily_speed.pivot(index="day", columns="violation_type", values="avg_speed")
        .ffill()
    )

    # Plot 1: Daily Violation Count 
    ax1.clear()
    for vtype in daily_pivot.columns:
        series = daily_pivot[vtype]
        roll   = series.rolling(7, min_periods=1).mean()
        color  = COLORS.get(vtype, "#555")
        ax1.bar(series.index, series.values, color=color, alpha=0.20, width=0.8)
        ax1.plot(roll.index, roll.values, color=color, linewidth=2,
                 label=f"{vtype} (7-day avg)")
        # MAX annotation
        idx_max = series.idxmax()
        ax1.annotate(
            f"MAX\n{series[idx_max]:.0f}",
            xy=(idx_max, series[idx_max]),
            xytext=(0, 12), textcoords="offset points",
            ha="center", fontsize=7, color=color, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=color, lw=1.2)
        )
        # P90 reference line
        p90 = series.quantile(0.90)
        ax1.axhline(p90, linestyle=":", color=color, alpha=0.5, linewidth=1)
        ax1.text(series.index[0], p90, f" P90={p90:.0f}", fontsize=6,
                 color=color, va="bottom")
    ax1.set_title("Daily Violation Count (by Arrival Time)", fontsize=9, fontweight="bold")
    ax1.set_xlabel("Date", fontsize=8)
    ax1.set_ylabel("Violations", fontsize=8)
    ax1.legend(fontsize=7)
    ax1.tick_params(labelsize=6, labelrotation=30)

    # Plot 2: Average Violation Speed
    ax2.clear()
    for vtype in speed_pivot.columns:
        series = speed_pivot[vtype].dropna()
        roll   = series.rolling(7, min_periods=1).mean()
        roll_std = series.rolling(7, min_periods=1).std().fillna(0)
        color  = COLORS.get(vtype, "#555")
        ax2.scatter(series.index, series.values, color=color, s=12, alpha=0.3)
        ax2.plot(roll.index, roll.values, color=color, linewidth=2,
                 label=f"{vtype} (7-day avg)")
        # Speed limit reference
        limit = REFERENCE_SPEED_LIMITS.get(vtype, 110)
        ax2.axhline(limit, linestyle="--", color=color, alpha=0.45, linewidth=1)
        ax2.text(series.index[0], limit,
                 f" {vtype} limit: {limit} km/h", fontsize=6, color=color)
        # SPIKE annotation (z-score > 2.5)
        z = (series - roll) / (roll_std + 1e-9)
        for idx, val in series[z > 2.5].items():
            ax2.annotate(
                f"SPIKE\n{val:.1f}",
                xy=(idx, val), xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=6, color="#e03131", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#e03131", lw=1.0)
            )
    ax2.set_title("Avg Violation Speed over Arrival Time (km/h)", fontsize=9, fontweight="bold")
    ax2.set_xlabel("Date", fontsize=8)
    ax2.set_ylabel("Speed (km/h)", fontsize=8)
    ax2.legend(fontsize=7)
    ax2.tick_params(labelsize=6, labelrotation=30)

    # Plot 3: Camera Breakdown 
    ax3.clear()
    cam_counts = (
        df.groupby(["camera_id_start", "violation_type"]).size()
        .reset_index(name="count")
    )
    cameras = sorted(cam_counts["camera_id_start"].unique())
    vtypes  = sorted(cam_counts["violation_type"].unique())
    x       = np.arange(len(cameras))
    width   = 0.35
    for i, vtype in enumerate(vtypes):
        sub    = cam_counts[cam_counts["violation_type"] == vtype]
        counts = [int(sub[sub["camera_id_start"] == c]["count"].sum()) for c in cameras]
        color  = COLORS.get(vtype, "#555")
        ax3.bar(x + i * width, counts, width, label=vtype, color=color, alpha=0.85)
        # Label bars
        for xi, cnt in zip(x + i * width, counts):
            ax3.text(xi, cnt + max(counts) * 0.01, str(cnt),
                     ha="center", fontsize=7, color=color)
        # Annotate highest
        max_idx = int(np.argmax(counts))
        ax3.annotate(
            f"HIGHEST\n{counts[max_idx]:,}",
            xy=(x[max_idx] + i * width, counts[max_idx]),
            xytext=(0, 18), textcoords="offset points",
            ha="center", fontsize=7, color="#e03131", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#e03131", lw=1.0)
        )
    ax3.set_xticks(x + width / 2)
    ax3.set_xticklabels([f"Cam {c}" for c in cameras], fontsize=8)
    ax3.set_title("Violations by Camera & Type", fontsize=9, fontweight="bold")
    ax3.set_ylabel("Count", fontsize=8)
    ax3.legend(fontsize=7)
    ax3.tick_params(labelsize=7)

    # Plot 4: Hourly Heatmap 
    ax4.clear()
    hourly = (
        df.groupby(["hour_of_day", "violation_type"]).size()
        .reset_index(name="count")
    )
    heatmap_data = (
        hourly.pivot(index="violation_type", columns="hour_of_day", values="count")
        .fillna(0)
        .reindex(columns=range(24), fill_value=0)
    )
    im = ax4.imshow(heatmap_data.values, aspect="auto", cmap="Blues")
    ax4.set_xticks(range(24))
    ax4.set_xticklabels(range(24), fontsize=5)
    ax4.set_yticks(range(len(heatmap_data.index)))
    ax4.set_yticklabels(heatmap_data.index, fontsize=7)
    ax4.set_title("Violations: Hour of Day × Type", fontsize=9, fontweight="bold")
    ax4.set_xlabel("Hour of Day (24h)", fontsize=8)
    # Annotate PEAK cell
    peak_loc = np.unravel_index(
        heatmap_data.values.argmax(), heatmap_data.values.shape
    )
    peak_val = int(heatmap_data.values[peak_loc])
    peak_hr  = heatmap_data.columns[peak_loc[1]]
    ax4.annotate(
        f"PEAK\n{peak_hr:02d}:00\n{peak_val}",
        xy=(peak_loc[1], peak_loc[0]),
        ha="center", va="center",
        fontsize=7, color="#e03131", fontweight="bold"
    )

    fig.canvas.draw()
    ts = pd.Timestamp.now().strftime("%H:%M:%S")
    print(f"[{ts}] Dashboard updated — {len(df):,} violations loaded from MongoDB.")


print("update_plots() defined.")

## Step 5 Start Real-Time Dashboard

The cell below initialises the figure and enters a polling loop.
MongoDB is queried every `POLL_INTERVAL` seconds and all four plots are refreshed with
the latest data.

**Stop the loop:** click the Stop button in the toolbar (raises `KeyboardInterrupt`).

In [ ]:
fig, ax1, ax2, ax3, ax4 = init_plots()

try:
    while True:
        update_plots(fig, ax1, ax2, ax3, ax4)
        sleep(POLL_INTERVAL)
except KeyboardInterrupt:
    print("\nDashboard stopped.")
    plt.close("all")

## Task 2.2.2 Interesting Point Annotation: Operational Insights

The dashboard annotates the following categories of notable points:

| Annotation | Definition | Operational Significance |
|------------|-----------|---------------------------|
| **MAX** | Day with the highest violation count (Plot 1) | Peak enforcement demand — pre-position resources on recurring high-count days |
| **P90 line** | 90th-percentile reference (Plot 1) | Days above this line warrant elevated patrol allocation |
| **SPIKE** | Day where speed is >2.5 std above 7-day rolling mean (Plot 2) | Statistically anomalous behaviour — warrants immediate investigation |
| **Speed limit line** | Legal limit per violation type (Plot 2) | Shows how far above the legal threshold detected violations sit on average |
| **HIGHEST** | Camera+type combination with the most violations (Plot 3) | Identifies the single most critical enforcement point |
| **PEAK** | Hour × type cell with the highest count (Plot 4) | Defines the optimal window for mobile enforcement deployment |

### Why These Visualisations Answer Operational Questions

**Plot 1 Daily Count Trend:**  
Enforcement managers can see which days consistently exceed the P90 threshold and
pre-allocate patrol resources. The INSTANTANEOUS / AVERAGE split distinguishes brief speed
lapses from sustained speeding, which is operationally more dangerous.

**Plot 2 Speed Pattern:**  
Tracking average violation speed over time reveals whether driver behaviour is deteriorating.
SPIKE annotations flag statistically anomalous days that need immediate follow-up.

**Plot 3 Camera Breakdown:**  
Camera-level aggregation reveals spatial hot-spots. If one camera consistently records
significantly more violations, it justifies a permanent enforcement presence or infrastructure
change (signage, rumble strips) at that location.

**Plot 4 Hourly Heatmap:**  
Time-of-day patterns enable *predictive* enforcement scheduling — mobile patrol units can be
deployed proactively during identified peak hours rather than reactively after incidents.